In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


---------------------------------------------------------------------------------------------

# MILESTONE-1

**Q1.)** Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. 
Based on your counts, what is the sum of the occurrences of the most frequent option 
and the least frequent option?

In [2]:
train_df=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [3]:
train_df['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most Frequent = B , freq = 490
--------------------------------
Least Frequent = E , freq = 324
--------------------------------



**Q2.)** After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [4]:
train_df.head()
X=train_df.copy()

In [5]:
import string 
vocab=set()
for text in X["prompt"].fillna(""):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    vocab.update(text.split())

print("Vocabulary size:", len(vocab))

Vocabulary size: 859


**Q3.)** Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering? 

In [6]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
pr=X['prompt'][0]
pr=pr.lower()
pr=pr.translate(str.maketrans("", "", string.punctuation))
words=pr.split()
filtered_words = [word for word in words if word not in ENGLISH_STOP_WORDS]
print((filtered_words))
print(f'number of words left after filtering : {len(filtered_words)}')

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
number of words left after filtering : 13


**Q4.)** Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
text = (
    X["prompt"].fillna("") + " " +
    X["A"].fillna("") + " " +
    X["B"].fillna("") + " " +
    X["C"].fillna("") + " " +
    X["D"].fillna("")
).tolist()

vectorizer=TfidfVectorizer(stop_words='english')
T=vectorizer.fit_transform(text)

print(T.shape[1])

2600


**Q5.)** Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

  

In [8]:
text = (
    X["prompt"].fillna("") + " " +
    X["A"].fillna("") + " " +
    X["B"].fillna("") + " " +
    X["C"].fillna("") + " " +
    X["D"].fillna("")
).tolist()

vectorizer=TfidfVectorizer(stop_words='english')
T=vectorizer.fit(text)

row = X.loc[X["id"] == 1].iloc[0]

prompt_vec = vectorizer.transform([row["prompt"]])
option_a_vec = vectorizer.transform([row["A"]])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0, 0]

print(similarity)

0.2766749714675434


**Q6.)** Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer. 

**Q9.)** The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [9]:
X['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [10]:
pred = ["B", "C", "A"]

def apk(actual, pred):
    if actual == pred[0]:
        return 1
    elif actual == pred[1]:
        return 1/2
    elif actual == pred[2]:
        return 1/3
    return 0

map3 = X["answer"].apply(lambda x: apk(x, pred)).mean()
print("MAP@3 =", map3)

MAP@3 = 0.42125


**Q10.)** The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [11]:
vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(text)

options = ["A", "B", "C", "D", "E"]

def map3_score(actual, preds):
    for i, pred in enumerate(preds[:3]):
        if pred == actual:
            return 1 / (i + 1)
    return 0

scores = []

for _, row in X.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = {}

    for option in options:
        option_vec = vectorizer.transform([row[option]])
        similarities[option] = cosine_similarity(
            prompt_vec, option_vec
        )[0, 0]

    top3 = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )[:3]

    scores.append(map3_score(row["answer"], top3))

final_map3 = sum(scores) / len(scores)

print("MAP@3 =", final_map3)

MAP@3 = 0.28475
